In [41]:
def evaluate_lof_scores(
    y_true,
    scores,
    experiment_name
):
    print("=" * 60)
    print(experiment_name)
    print("=" * 60)

    print(
        "score_samples ROC-AUC:",
        roc_auc_score(
            y_true,
            scores
        )
    )

    print(
        "-score_samples ROC-AUC:",
        roc_auc_score(
            y_true,
            -scores
        )
    )

    print(
        "score_samples PR-AUC:",
        average_precision_score(
            y_true,
            scores
        )
    )

    print(
        "-score_samples PR-AUC:",
        average_precision_score(
            y_true,
            -scores
        )
    )

In [32]:
print("⟳ Loading Random Forest model...")
best_rf = joblib.load("models/random_forest/random_forest_best_randomsplit.joblib")
print("✅ Done.\n")

rf_probs = best_rf.predict_proba(X_test_rf)[:, 1]
rf_preds = best_rf.predict(X_test_rf)

print("⚖️ Evaluation of Random Forest model with random split(Experiment 1)...")
print(f"Accuracy : {accuracy_score(y_test_rf, rf_preds):.4f}")
print(f"Precision: {precision_score(y_test_rf, rf_preds):.4f}")
print(f"Recall   : {recall_score(y_test_rf, rf_preds):.4f}")
print(f"F1-score : {f1_score(y_test_rf, rf_preds):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test_rf, rf_probs):.4f}")
print(f"PR-AUC   : {average_precision_score(y_test_rf, rf_probs):.4f}\n")

print("Confusion matrix:")
cm = confusion_matrix(y_test_rf, rf_preds)
print(cm)

print("Classification report:")
print(classification_report(y_test_rf, rf_preds, digits=4))

print("Feature importance:")
rf_search = joblib.load("models/random_forest/rf_random_search_randomsplit.joblib")
feature_importance = (
    pd.DataFrame({
        "feature": X_train_rf.columns,
        "importance": best_rf.named_steps["model"].feature_importances_
    }).sort_values("importance", ascending=False)
)
print(f"{feature_importance.head(20)}\n")

personal_values = feature_importance.loc[feature_importance["feature"].str[1:].astype(int) < 95,
    "importance"
    ].tolist()
print(f"Aggregated personal features' importance value: {sum(personal_values):.4f}\nAggregated neighborhood graph features' importance value: {1-sum(personal_values):.4f}\n")

file_path = "models/random_forest/rf_random_search_results_randomsplit.csv"
if not file_path:
    print("Saving the tuning results...")
    cv_results = pd.DataFrame(rf_search.cv_results_)
    cv_results = cv_results.sort_values(
        "rank_test_score"
    )
    cv_results.to_csv("models/random_forest/rf_random_search_results.csv", index=False)
    

⟳ Loading Random Forest model...
✅ Done.

⚖️ Evaluation of Random Forest model with random split(Experiment 1)...
Accuracy : 0.9902
Precision: 0.9812
Recall   : 0.9175
F1-score : 0.9483
ROC-AUC  : 0.9972
PR-AUC   : 0.9842

Confusion matrix:
[[8388   16]
 [  75  834]]
Classification report:
              precision    recall  f1-score   support

           0     0.9911    0.9981    0.9946      8404
           1     0.9812    0.9175    0.9483       909

    accuracy                         0.9902      9313
   macro avg     0.9862    0.9578    0.9714      9313
weighted avg     0.9902    0.9902    0.9901      9313

Feature importance:
    feature  importance
52      f54    0.114984
4        f6    0.098554
13      f15    0.087939
54      f56    0.071851
89      f91    0.040306
162    f164    0.037538
40      f42    0.030103
64      f66    0.024218
58      f60    0.024190
131    f133    0.023910
159    f161    0.019922
15      f17    0.019390
60      f62    0.018203
46      f48    0.017777
17

In [46]:
print("҉ Loading Isolation Forest model...")
isolation_forest_model = joblib.load("models/isolation_forest/isolation_forest_best.joblib")
print("✅ Loaded successfully!\n")
print("🔄 Generate final test predictions")
test_predictions_df = create_if_predictions(
    model=isolation_forest_model,
    X=X_test,
    df=test_df
)


test_predictions_df.head()

҉ Loading Isolation Forest model...
✅ Loaded successfully!

🔄 Generate final test predictions


,txId,time_step,anomaly_score,decision_function,prediction
0,72340883,42,0.413166,0.045693,0
1,73005229,42,0.596482,-0.137623,1
2,73243693,42,0.356850,0.102009,0
3,73243691,42,0.357684,0.101175,0
4,3866934,42,0.388741,0.070118,0


In [48]:
print("==============================")
print("🏆 FINAL TEST IF RANKING")
print("==============================")


y_true = test_eval_df["is_illicit"]

y_score = test_eval_df["anomaly_score"]

print(
    "ROC-AUC:",
    roc_auc_score(
        y_true,
        y_score
    )
)

print(
    "PR-AUC:",
    average_precision_score(
        y_true,
        y_score
    )
)

🏆 FINAL TEST IF RANKING
ROC-AUC: 0.7191210766293947
PR-AUC: 0.28734625439743194


In [67]:
# Final LOF — Test Evaluation
# ============================================================
# SCORE TEST PERIOD
# ============================================================

X_test_lof_final = (
    test_if_df[feature_columns_unsupervised]
    .copy()
)

test_anomaly_scores = -final_lof.score_samples(
    X_test_lof_final
)

test_predictions = pd.DataFrame({
    "txId": test_df["txId"].values,
    "time_step": test_df["time_step"].values,
    "anomaly_score": test_anomaly_scores,
})

print(
    "Test transactions scored:",
    len(test_predictions)
)

display(test_predictions.head())

# ============================================================
# CREATE TEST EVALUATION DATASET
# ============================================================

test_eval = test_predictions.merge(
    test_labels,
    on="txId",
    how="inner",
)

print(
    "All test transactions:",
    len(test_predictions)
)

print(
    "Labeled test transactions:",
    len(test_eval)
)

print("\nClass distribution:")
print(
    test_eval["is_illicit"].value_counts()
)

display(test_eval.head())

# ============================================================
# FINAL LOF TEST PERFORMANCE
# ============================================================

y_test_true = (
    test_eval["is_illicit"]
    .astype(int)
)

y_test_score = (
    test_eval["anomaly_score"]
    .astype(float)
)

test_roc_auc = roc_auc_score(
    y_test_true,
    y_test_score,
)

test_pr_auc = average_precision_score(
    y_test_true,
    y_test_score,
)

print("=" * 60)
print("FINAL LOF — TEST PERFORMANCE")
print("=" * 60)

print(f"ROC-AUC: {test_roc_auc:.6f}")
print(f"PR-AUC:  {test_pr_auc:.6f}")

Test transactions scored: 36824


,txId,time_step,anomaly_score
0,72340883,42,1.228538
1,73005229,42,1.985996
2,73243693,42,1.032942
3,73243691,42,1.205880
4,3866934,42,1.534561


All test transactions: 36824
Labeled test transactions: 8841

Class distribution:
is_illicit
0    8433
1     408
Name: count, dtype: int64


,txId,time_step,anomaly_score,is_illicit
0,73005229,42,1.985996,0
1,99949697,42,1.259651,0
2,72306537,42,1.043084,0
3,79428037,42,1.640983,0
4,73028219,42,1.241553,0


FINAL LOF — TEST PERFORMANCE
ROC-AUC: 0.460650
PR-AUC:  0.047858
